# 14 · Performance & Cost

Making pipelines **fast and cheap** is core data-engineering craft. This notebook
covers the levers that matter most on Databricks — how Spark executes your code,
shuffles and joins, file layout (partitioning vs clustering), caching, and the
platform features (AQE, Photon) — plus cost habits.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F
spark.sql("USE SCHEMA brewbox")
print("ready")

## 1 · Lazy execution & the query plan

Transformations (`select`, `filter`, `join`, `groupBy`) are **lazy** — they build
a plan. Only an **action** (`count`, `show`, `write`, `collect`) runs it. Spark's
**Catalyst** optimizer rewrites the plan; **`explain()`** shows the physical plan
it will execute — your first tuning tool.

In [ ]:
q = (spark.table("brewbox.fct_orders").filter("is_completed")
     .groupBy("store_id").agg(F.sum("amount").alias("rev")))
q.explain()          # read bottom-up: scan -> filter -> exchange (shuffle) -> aggregate

## 2 · Shuffles & broadcast joins

The most expensive thing in Spark is a **shuffle** — moving data across the
network (joins and `groupBy` cause them). When you join a **big** table to a
**small** one, **broadcast** the small side: Spark ships it to every executor, so
no big shuffle is needed.

In [ ]:
from pyspark.sql.functions import broadcast
big = spark.table("brewbox.fct_orders")
small = spark.table("brewbox.dim_store")

joined = big.join(broadcast(small), "store_id")   # broadcast the small dimension
joined.explain()                                   # note "BroadcastHashJoin" in the plan
print("rows:", joined.count())

Databricks' **Adaptive Query Execution (AQE)** auto-detects small sides and
converts joins to broadcasts at runtime, so you often don't need to hint — but
knowing the pattern helps you read plans and fix stubborn cases.

## 3 · File layout — partitioning vs Z-order vs liquid clustering

How data is laid out on disk decides how much a query can **skip** (data
skipping):

- **Partitioning** (`PARTITIONED BY (col)`) — physically splits files by a
  **low-cardinality** column (e.g., `date`). Great for filters on that column;
  bad if over-partitioned (the small-file problem).
- **Z-ordering** (`OPTIMIZE t ZORDER BY (col)`) — co-locates related rows within
  files so queries skip more files. Good for high-cardinality filter columns.
- **Liquid clustering** (`CLUSTER BY (col)`) — the newer, automatic replacement
  for both; you set clustering keys and Databricks maintains layout as data
  changes. Prefer it for new tables.

And always fight the **small-file problem** with `OPTIMIZE` (compaction).

In [ ]:
# Compact + cluster the Silver table for faster reads (Delta)
spark.sql("OPTIMIZE brewbox.orders_silver ZORDER BY (order_date)")
spark.sql("DESCRIBE DETAIL brewbox.orders_silver").select("numFiles","sizeInBytes").show()

## 4 · Caching

If you reuse a DataFrame many times in a session, **cache** it so Spark doesn't
recompute it each time. Cache only what you reuse (it costs memory), and
`unpersist` when done.

In [ ]:
completed = spark.table("brewbox.fct_orders").filter("is_completed").cache()
print("count (materializes cache):", completed.count())     # first action fills cache
print("reused from cache:", completed.agg(F.sum("amount")).first()[0])
completed.unpersist()

## 5 · Platform accelerators & skew

- **Photon** — Databricks' vectorized C++ engine; transparently speeds up SQL/
  DataFrame workloads. Enable it on your compute; no code change.
- **AQE** — adaptive shuffle partitions, broadcast conversion, and **skew join**
  handling, on by default.
- **Data skew** — one key with far more rows than others makes one task the
  bottleneck. AQE mitigates it; you can also salt keys or filter hot keys.

## 6 · Cost habits

- Use **serverless** or **job clusters** with **auto-termination** — never leave
  compute idle.
- **Right-size** compute; bigger isn't always faster (shuffles dominate).
- Filter & `select` **early** to shrink data before joins/aggregations.
- Avoid `collect()` / `toPandas()` on large data (pulls everything to the driver).
- Prefer **incremental** processing (Auto Loader, CDF, MERGE) over full rebuilds.
- Store analytics data as **Delta/Parquet** (columnar) and keep files ~128MB–1GB
  via `OPTIMIZE`.

## 7 · Exercises

**Exercise 1 —** Use `explain()` on a join of `fct_order_items` to `dim_product`
and confirm whether Spark chose a broadcast join.

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
(spark.table("brewbox.fct_order_items")
    .join(spark.table("brewbox.dim_product"), "product_id")
    .explain())

**Exercise 2 —** Run `OPTIMIZE` on `brewbox.fct_orders` and report the number of
files before and after (hint: `DESCRIBE DETAIL`).

In [ ]:
# Your turn (Exercise 2):

In [ ]:
# ✅ Solution 2
before = spark.sql("DESCRIBE DETAIL brewbox.fct_orders").first()["numFiles"]
spark.sql("OPTIMIZE brewbox.fct_orders")
after = spark.sql("DESCRIBE DETAIL brewbox.fct_orders").first()["numFiles"]
print("files before:", before, "-> after:", after)

**Exercise 3 —** Name two reasons to avoid `df.toPandas()` on a large table.
(Answer in the cell.)

In [ ]:
print("1) It pulls ALL rows to the single driver machine (can OOM / crash).")
print("2) It abandons distributed execution - the work no longer scales.")

## 8 · Recap & next

Spark is lazy (actions trigger plans — read them with `explain`); **shuffles** are
costly, so **broadcast** small dimensions; lay data out with **partitioning /
Z-order / liquid clustering** and `OPTIMIZE`; **cache** reuse; lean on **Photon**
and **AQE**; and keep compute right-sized and auto-terminating for cost.

**Next → `15` Capstone:** put the whole track together into one end-to-end,
production-shaped BrewBox pipeline. 🏁